LoRA Fine-Tuning Orchestrator

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected - connect a Colab GPU kernel first"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(props.total_memory / 1e9, 2))
print("torch:", torch.__version__)

Phase 0 — Environment Verification

Verifies Environment: Confirms dependencies and GPU environment are fully set up and creates environment log.

In [ ]:
!python TASK1_finetuning_model/scripts/env_check.py

In [ ]:
!pip freeze > TASK1_finetuning_model/environment.txt
print("Wrote TASK1_finetuning_model/environment.txt - commit this back to the repo (reproducibility lock).")

Phase 1 — Data Extraction & Cleaning

Extracts & Cleans Text: Reads raw PDF documents, extracts text, cleans layout noise, and outputs baseline dataset and metadata.

In [ ]:
!python TASK1_finetuning_model/scripts/extract_text.py

In [ ]:
# Quick sanity check: first & last 300 chars + stats summary
with open('data/extracted/document_clean.txt', encoding='utf-8') as f:
    text = f.read()
print('--- FIRST 300 CHARS ---')
print(text[:300])
print('\n--- LAST 300 CHARS ---')
print(text[-300:])

import json
stats = json.load(open('data/extracted/stats.json'))
print('\n--- STATS ---')
for k, v in stats.items():
    if k != 'note':
        print(f'  {k}: {v}')

# Show any garbled pages
manifest = json.load(open('data/extracted/extraction_manifest.json'))
garbled = [m for m in manifest if m['garbled_flag']]
if garbled:
    print(f'\n⚠ {len(garbled)} page(s) flagged as garbled:')
    for m in garbled:
        print(f"  page {m['page']:3d}  chosen={m['extractor_chosen']}  reason={m['reason']}")
        print(f"          non_ascii={m['final_scores']['non_ascii_ratio']:.1%}  "
              f"non_dict={m['final_scores']['non_dict_word_ratio']:.1%}")
else:
    print('\n✓ No pages flagged as garbled.')

Phase 2 — Base Model & Tokenizer Selection

Loads Model & Tokenizer: Sets up pad tokens and calculates exact token count for dataset.

Maps Layer Names: Saves model layer names to configs/model_architecture.json for LoRA layer targeting.

Checks Memory Needs: Confirms 135M model fits in GPU memory without requiring 4-bit compression.

In [ ]:
!python TASK1_finetuning_model/scripts/select_model.py

In [ ]:
# Phase 2 post-run inspection
import json

# 1. Updated token counts
stats = json.load(open('data/extracted/stats.json'))
print('--- TOKEN COUNTS ---')
print(f"  proxy_token_count_gpt2_tiktoken : {stats.get('proxy_token_count_gpt2_tiktoken')}")
print(f"  exact_token_count_smollm2_135m  : {stats.get('exact_token_count_smollm2_135m')}")

# 2. Run config summary
cfg = json.load(open('TASK1_finetuning_model/configs/run_phase2.json'))
print('\n--- RUN CONFIG SUMMARY ---')
for k in ('model_name', 'total_params', 'pad_token', 'quantization_needed',
          'fp16_weight_footprint_mb', 'vram_utilisation_weights_only_pct'):
    print(f"  {k}: {cfg.get(k)}")

# 3. Architecture spot-check — attention projection layers for Phase 4
arch = json.load(open('TASK1_finetuning_model/configs/model_architecture.json'))
attn_kw = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'c_attn')
attn = [n for n in arch['all_module_names'] if any(k in n for k in attn_kw)]
print(f"\n--- ATTENTION MODULES ({len(attn)} found — first 10) ---")
for name in attn[:10]:
    print(f"  {name}")


Phase 3 — Dataset Construction (Chunking & Splitting)

Contiguous 85/15 Split: Splits tokenized document into 85% train and 15% validation sets to prevent data leakage.

Asymmetric Chunks: Creates 256-token chunks with 50% overlap for training and 0% overlap for validation.

Saves PyTorch Tensors: Outputs finetune_train.pt and finetune_val.pt tensors along with dataset statistics logs.

In [ ]:
!python TASK1_finetuning_model/scripts/build_dataset.py

In [ ]:
# Phase 3 post-run inspection
import json, torch

# 1. Dataset stats
stats = json.load(open('data/processed/dataset_stats.json'))
print('--- DATASET STATS ---')
for k, v in stats.items():
    if k not in ('model_name', 'timestamp', 'note'):
        print(f'  {k}: {v}')

# 2. Tensor shapes (weights_only=True is clean -- pure tensor dict, no custom classes)
train_data = torch.load('data/processed/finetune_train.pt', weights_only=True)
val_data   = torch.load('data/processed/finetune_val.pt',   weights_only=True)
print('\n--- TENSOR SHAPES ---')
print('  train input_ids:', tuple(train_data['input_ids'].shape))
print('  val   input_ids:', tuple(val_data['input_ids'].shape))

# 3. Spot-check: decode first train and first val chunk
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-135M')
print('\n--- TRAIN CHUNK[0] (first 80 tokens decoded) ---')
print(tok.decode(train_data['input_ids'][0][:80]))
print('\n--- VAL CHUNK[0] (first 80 tokens decoded) ---')
print(tok.decode(val_data['input_ids'][0][:80]))


Phase 4 — LoRA Configuration & Model Wrapping

Configures & Wraps LoRA: Applies LoRA adapters (r=8, alpha=16) to attention modules, freezing base weights for parameter-efficient fine-tuning.

Generates Rank Sweeps: Writes 3 sweep configs (r=4, r=8, r=16) in configs/ for training runs.

Passes Sanity Check: Runs 1-batch forward and backward pass to confirm gradient flow and logs trainable parameters to configs/trainable_params.json.

In [ ]:
!python TASK1_finetuning_model/scripts/wrap_lora.py

In [ ]:
# Phase 4 post-run inspection
import json, os

params = json.load(open('TASK1_finetuning_model/configs/trainable_params.json'))
print('--- TRAINABLE PARAMS ---')
for k, v in params.items():
    if k != 'note':
        print(f'  {k}: {v}')

# Spot-check: trainable % should be <<10% for LoRA
pct = params.get('trainable_percentage', 0)
if 0 < pct < 10:
    print(f'\n  trainable_percentage={pct:.3f}% -- looks right for LoRA r=8')
else:
    print(f'\n  trainable_percentage={pct:.3f}% -- UNEXPECTED. Check target_modules.')

# Check sweep configs were written
for r in [4, 8, 16]:
    p = f'TASK1_finetuning_model/configs/run_phase4_r{r}.json'
    status = 'OK' if os.path.exists(p) else 'MISSING'
    print(f'  sweep config r={r}: {status} ({p})')

# Sanity loss
loss = params.get('sanity_check_loss')
finite = params.get('sanity_check_loss_finite')
print(f'\n  sanity_check_loss: {loss}  (finite: {finite})')


Phase 5 — Training Loop & Execution

Manual PyTorch Loop: Executes explicit training loop with AdamW, cosine learning rate decay, and gradient clipping (1.0).

AMP & Stability: Uses automatic mixed precision (autocast + GradScaler) to prevent gradient underflow.

Sweeps 3 Ranks: Runs 3 independent rank sweeps (r=4, r=8, r=16) and saves best validation adapters.

Logs & Early Stops: Appends step metrics to logs/<run>/metrics.jsonl and summarizes sweep results in eval/sweep_results.csv.

In [ ]:
# Sweep run 1 — LoRA r=4
!python TASK1_finetuning_model/scripts/train.py --run r4

In [ ]:
# Sweep run 2 — LoRA r=8 (plan default)
!python TASK1_finetuning_model/scripts/train.py --run r8

In [ ]:
# Sweep run 3 — LoRA r=16
!python TASK1_finetuning_model/scripts/train.py --run r16

In [ ]:
# Phase 5 — post-sweep inspection
import csv, os
csv_path = 'TASK1_finetuning_model/eval/sweep_results.csv'
if os.path.exists(csv_path):
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    print(f'Sweep results ({len(rows)} runs):')
    for r in rows:
        print(
            f"  {r['run_name']:6s}  "
            f"best_val={float(r['best_val_loss']):.4f}  "
            f"final_val={float(r['final_val_loss']):.4f}  "
            f"best_epoch={r['best_val_epoch']}"
        )
    best = min(rows, key=lambda x: float(x['best_val_loss']))
    print(f"\nBest run: {best['run_name']} (best_val_loss={float(best['best_val_loss']):.4f})")
else:
    print(f'sweep_results.csv not found at {csv_path} — run all 3 training cells first.')


Phase 6 — Quantitative Evaluation

Selects Best Checkpoint: Identifies top-performing sweep run from eval/sweep_results.csv and loads its best checkpoint.

Plots Loss Curves: Generates eval/loss_curve.png displaying training vs. validation loss across all steps.

Calculates Core Metrics: Computes cross-entropy loss, perplexity, and Bits-Per-Byte (BPB), saving output to eval/final_metrics.json.

In [ ]:
# Phase 6 — evaluate best checkpoint (auto-picks from sweep_results.csv)
!python TASK1_finetuning_model/scripts/evaluate.py

In [ ]:
# Phase 6 — display final metrics and embed loss curve
import json
from IPython.display import Image, display

metrics = json.load(open('TASK1_finetuning_model/eval/final_metrics.json'))
print('--- FINAL METRICS ---')
for k, v in metrics.items():
    if k not in ('bpb_note', 'timestamp', 'checkpoint_dir'):
        print(f'  {k}: {v}')

print(f"\n  BPB = {metrics['bpb']}  (cross-track comparable metric)")
print(f"  Perplexity = {metrics['perplexity']}")

img_path = 'TASK1_finetuning_model/eval/loss_curve.png'
import os
if os.path.exists(img_path):
    display(Image(img_path))
else:
    print(f'loss_curve.png not found at {img_path}')


Phase 7 — Qualitative Evaluation (Generation)

Extracts Held-Out Prompts: Deterministically extracts 8 prompt prefixes from unseen validation region.

Generates Dual Completions: Uses sampling and greedy decoding modes to generate completions for each prompt.

Outputs Samples: Saves prompt and completion pairs to generations/finetuning_samples.md for qualitative review.

In [ ]:
# Phase 7 — generate completions from best checkpoint
!python TASK1_finetuning_model/scripts/generate.py

In [ ]:
# Phase 7 — preview first 3 samples from finetuning_samples.md
import os
md_path = 'TASK1_finetuning_model/generations/finetuning_samples.md'
if os.path.exists(md_path):
    text = open(md_path, encoding='utf-8').read()
    # Print up to the 4th sample marker to preview without flooding output
    markers = [i for i, line in enumerate(text.splitlines()) if line.startswith('## Sample')]
    cutoff = text.splitlines()[markers[3]] if len(markers) >= 4 else text.splitlines()[-1]
    end_idx = text.find(cutoff) if len(markers) >= 4 else len(text)
    print(text[:end_idx])
    print(f'... ({len(markers)} total samples in {md_path})')
else:
    print(f'{md_path} not found — run generate.py first.')


Phase 8 — Cross-Track Comparison Prep

Stages Track 1 Artifacts: Copies metric JSONs and loss curve plots to shared_eval with finetuning_* namespacing.

Authors Comparison Matrix: Writes shared_eval/comparison_notes.md pre-filled with Track 1's BPB, perplexity, and parameter efficiency numbers.

Stubs Track 2: Leaves Track 2 fields as TBD so Track 1 deliverables are staged immediately without waiting for Track 2.

In [ ]:
# Phase 8 — stage Track 1 artifacts for cross-track comparison
!python TASK1_finetuning_model/scripts/stage_comparison.py

In [ ]:
# Phase 8 — verify shared_eval/ contents
import os
shared = 'shared_eval'
if os.path.exists(shared):
    files = os.listdir(shared)
    print(f'shared_eval/ contents ({len(files)} files):')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(shared, f))
        print(f'  {f}  ({size:,} bytes)')
else:
    print(f'{shared}/ not found — run stage_comparison.py first.')


In [ ]:
TOKEN = "token"

!git config --global user.name "DevaNandanJS"
!git config --global user.email "devanandan@example.com"
!git add -A
!git commit -m "sync colab run outputs"
!git push https://{TOKEN}@github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git main
